# BÀI TẬP: MEDICAL INSURANCE COST
**Nguồn:** kaggle.com/datasets/mosapabdelghany/medical-insurance-cost-dataset



## Setup

In [1]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from scipy import stats

sns.set_style('whitegrid')


csv_path = 'https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv'

df = pd.read_csv(csv_path)
print('Loaded from:', csv_path)
df.head()

Loaded from: https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


---
# PHẦN A — DATA PROFILING
## A.1. Data size, column names, data types

In [2]:
# TODO
# TODO
print('Số hàng:', df.shape[0], 'Số cột:', df.shape[1])
print()
print('Columns:',df.columns.tolist())
print()
print('Data types:', df.dtypes)


Số hàng: 1338 Số cột: 7

Columns: ['age', 'sex', 'bmi', 'children', 'smoker', 'region', 'charges']

Data types: age           int64
sex          object
bmi         float64
children      int64
smoker       object
region       object
charges     float64
dtype: object


## A.2. Missing values & Duplicate data

In [3]:
# TODO
# TODO
missing = df.isnull().sum()
print("Missing values:")
print(missing[missing>0] if missing.sum()>0 else "Không có cột nào thiếu dữ liệu.")
print()
print('Duplicate values:', df.duplicated().sum())

Missing values:
Không có cột nào thiếu dữ liệu.

Duplicate values: 1


## A.3. Invalid values

In [4]:
# TODO
print("AGE < 0:", (df['age'] < 0).sum())
print("BMI < 0:", (df['bmi'] < 0).sum())
print("CHILDREN < 0:", (df['children'] < 0).sum())
print("CHARGES < 0:", (df['charges'] < 0).sum())
print()
print("=> Kết luận: Dữ liệu đã được kiểm tra tính hợp lý.")

AGE < 0: 0
BMI < 0: 0
CHILDREN < 0: 0
CHARGES < 0: 0

=> Kết luận: Dữ liệu đã được kiểm tra tính hợp lý.


## A.4. Create a new column
Tạo cột `bmi_group`: Normal (<25), Overweight (25-30), Obese (>=30).

In [7]:
# TODO
def bmi_g(bmi):
    if bmi < 25:
        return 'Normal'
    elif bmi < 30:
        return 'Overweight'
    else:
        return 'Obese'

df['bmi_group'] = df['bmi'].apply(bmi_g)
df.head()

,age,sex,bmi,children,smoker,region,charges,bmi_group
0,19,female,27.900,0,yes,southwest,16884.92400,Overweight
1,18,male,33.770,1,no,southeast,1725.55230,Obese
2,28,male,33.000,3,no,southeast,4449.46200,Obese
3,33,male,22.705,0,no,northwest,21984.47061,Normal
4,32,male,28.880,0,no,northwest,3866.85520,Overweight


---
# PHẦN B — DESCRIPTIVE STATISTICS
## Group 1 — Central Tendency

In [8]:
# TODO
for col in ['age', 'bmi', 'children', 'charges']: 
    s = df[col].dropna() 
    mean, median, mode = s.mean(), s.median(), s.mode()[0]
    print(f"{col.upper()}: Mean={mean:.2f}, Median={median:.2f}, Mode={mode:.2f}")

AGE: Mean=39.21, Median=39.00, Mode=18.00
BMI: Mean=30.66, Median=30.40, Mode=32.30
CHILDREN: Mean=1.09, Median=1.00, Mode=0.00
CHARGES: Mean=13270.42, Median=9382.03, Mode=1639.56


## Group 2 — Dispersion

In [9]:
# TODO
for col in ['age', 'bmi', 'charges']:
    s = df[col].dropna()
    range_ = s.max() - s.min()
    std_ = s.std()
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    cv = std_ / s.mean()
    print(f"--- {col.upper()} ---")
    print(f"Range={range_:.2f}, Std={std_:.2f}, IQR={iqr:.2f}, CV={cv:.2f}\n")

--- AGE ---
Range=46.00, Std=14.05, IQR=24.00, CV=0.36

--- BMI ---
Range=37.17, Std=6.10, IQR=8.40, CV=0.20

--- CHARGES ---
Range=62648.55, Std=12110.01, IQR=11899.63, CV=0.91



## Group 3 — Location and Shape

In [10]:
# TODO
# TODO
print(df[["age", "bmi", "children", "charges"]].describe())

print("\nSkewness:")
print(df[["age", "bmi", "children", "charges"]].skew())

               age          bmi     children       charges
count  1338.000000  1338.000000  1338.000000   1338.000000
mean     39.207025    30.663397     1.094918  13270.422265
std      14.049960     6.098187     1.205493  12110.011237
min      18.000000    15.960000     0.000000   1121.873900
25%      27.000000    26.296250     0.000000   4740.287150
50%      39.000000    30.400000     1.000000   9382.033000
75%      51.000000    34.693750     2.000000  16639.912515
max      64.000000    53.130000     5.000000  63770.428010

Skewness:
age         0.055673
bmi         0.284047
children    0.938380
charges     1.515880
dtype: float64


---
# PHẦN C — DEFINE THE QUESTION

## Câu hỏi 1: Người hút thuốc trả chi phí cao hơn bao nhiêu lần so với người không hút, và có đồng đều giữa các vùng không?

In [14]:
# TODO
hutthuoc = df.groupby('smoker')['charges'].mean()
cohut = hutthuoc['yes'] / hutthuoc['no']
print(f"Chi phí trung bình - Hút thuốc: {hutthuoc['yes']:.2f}")
print(f"Chi phí trung bình - Không hút: {hutthuoc['no']:.2f}")
print(f"Người hút thuốc trả cao hơn khoảng {cohut:.1f} lần.")
print()

pivot_smoker = df.pivot_table(
    index='region',      
    columns='smoker',     
    values='charges',  
    aggfunc='mean',   
    fill_value=0     
).reset_index()

print("Chi phí theo vùng và trạng thái hút thuốc:")
print(pivot_smoker.round(2))

Chi phí trung bình - Hút thuốc: 32050.23
Chi phí trung bình - Không hút: 8434.27
Người hút thuốc trả cao hơn khoảng 3.8 lần.

Chi phí theo vùng và trạng thái hút thuốc:
smoker     region       no       yes
0       northeast  9165.53  29673.54
1       northwest  8556.46  30192.00
2       southeast  8032.22  34845.00
3       southwest  8019.28  32269.06


## Câu hỏi 2: BMI có tương quan với chi phí mạnh hơn ở nhóm hút thuốc hay không hút thuốc?

In [15]:
# TODO
smoker_yes = df[df['smoker'] == 'yes']
smoker_no = df[df['smoker'] == 'no']

corr_yes = smoker_yes['bmi'].corr(smoker_yes['charges'])
corr_no = smoker_no['bmi'].corr(smoker_no['charges'])

print(f"Hệ số tương quan (BMI vs Charges) ở nhóm HÚT THUỐC: {corr_yes:.3f}")
print(f"Hệ số tương quan (BMI vs Charges) ở nhóm KHÔNG HÚT THUỐC: {corr_no:.3f}")

Hệ số tương quan (BMI vs Charges) ở nhóm HÚT THUỐC: 0.806
Hệ số tương quan (BMI vs Charges) ở nhóm KHÔNG HÚT THUỐC: 0.084


## Câu hỏi 3: Vùng nào có chi phí bảo hiểm trung bình cao nhất?

In [19]:
# TODO
region_charges = df.groupby('region')['charges'].mean().sort_values(ascending=False)
print("Chi phí bảo hiểm trung bình theo vùng:")
print(region_charges.round(2))
print()
print("Nhận xét: Vùng southeast có chi phí bảo hiểm trung bình cao nhất.")

Chi phí bảo hiểm trung bình theo vùng:
region
southeast    14735.41
northeast    13406.38
northwest    12417.58
southwest    12346.94
Name: charges, dtype: float64

Nhận xét: Vùng southeast có chi phí bảo hiểm trung bình cao nhất.


## Câu hỏi 4: Số lượng con cái có làm tăng chi phí bảo hiểm không?

In [22]:
# TODO
children_charges = df.groupby('children')['charges'].mean()
print("Chi phí bảo hiểm trung bình theo số lượng con:")
print(children_charges.round(2))
print()
print("Số lượng con cái không làm tăng chi phí bảo hiểm một cách rõ rệt, nhưng khi có từ 4 con trở lên chi phí bắt đầu giảm")

Chi phí bảo hiểm trung bình theo số lượng con:
children
0    12365.98
1    12731.17
2    15073.56
3    15355.32
4    13850.66
5     8786.04
Name: charges, dtype: float64

Số lượng con cái không làm tăng chi phí bảo hiểm một cách rõ rệt, nhưng khi có từ 4 con trở lên chi phí bắt đầu giảm


## Câu hỏi 5: Tuổi có tương quan với chi phí bảo hiểm không?

In [23]:
# TODO
corr_age = df['age'].corr(df['charges'])
print(f"Hệ số tương quan giữa Tuổi (Age) và Chi phí (Charges): {corr_age:.3f}")
print()
print("Nhận xét: Tuổi có tương quan đồng biến mạnh với chi phí bảo hiểm (tuổi càng cao, chi phí càng lớn).")

Hệ số tương quan giữa Tuổi (Age) và Chi phí (Charges): 0.299

Nhận xét: Tuổi có tương quan đồng biến mạnh với chi phí bảo hiểm (tuổi càng cao, chi phí càng lớn).


## Câu hỏi 6 (Tổng hợp) — Viết insight tổng hợp
Dựa trên Phần A, B, C, viết 4-5 câu insight tổng thể.

Yếu tố chi phối lớn nhất: Hút thuốc là nguyên nhân làm tăng chi phí y tế vượt trội nhất (gấp nhiều lần so với người không hút).

Tác động của BMI: Chỉ số BMI cao kết hợp với việc hút thuốc đẩy chi phí bảo hiểm lên mức cao rõ rệt.

Ảnh hưởng của tuổi tác: Tuổi tác có mối quan hệ đồng biến mạnh, tuổi càng cao thì chi phí bảo hiểm càng tăng.

Các yếu tố ít tác động hơn: Số lượng con cái và vùng địa lý không tạo ra sự chênh lệch chi phí lớn bằng các yếu tố về hành vi cá nhân như hút thuốc hay thể trạng.